# Bronze to Silver — CineData Analytics

**Regra da camada Silver:** as tabelas Bronze não são alteradas em nenhum momento —
elas são apenas lidas. Toda a limpeza, tipagem e renomeação para português acontece aqui,
gerando tabelas novas no database `silver`.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "workspace"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql("CREATE DATABASE IF NOT EXISTS silver")

## 1) silver.tb_info_filmes
Origem: `bronze.tb_movies_info`

Tratamentos aplicados:
1. Deduplicação por filme, mantendo a linha de ingestão mais recente.
2. Normalização e tradução da coluna de status.
3. Conversão da data de lançamento testando os 3 formatos presentes na origem.
4. Criação da coluna derivada `ano_lancamento`.

In [0]:
df_bronze_info = spark.table("bronze.tb_movies_info")

print(f"Linhas na Bronze: {df_bronze_info.count()}")
print(f"Filmes distintos: {df_bronze_info.select('id').distinct().count()}")

### 1.1 Deduplicação
A Bronze grava em modo append, então o mesmo filme pode aparecer mais de uma vez
(seja por duplicidade na origem, seja por reexecução do pipeline).
A regra é manter apenas a versão mais recente, e é para isso que existe a coluna
`ingestion_datetime`: ela ordena as versões do mesmo `id`.

In [0]:
janela_filme = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())

df_dedup = (
    df_bronze_info
    .withColumn("num_versao", F.row_number().over(janela_filme))
    .filter(F.col("num_versao") == 1)
    .drop("num_versao")
)

print(f"Linhas após deduplicação: {df_dedup.count()}")

### 1.2 Normalização e tradução do status
A origem traz o mesmo status escrito de várias formas: `Released`, `RELEASED`, `released`,
`In-Production`, `In Production`. Existem ainda registros corrompidos (datas no lugar do status).

Por isso a normalização vem **antes** da tradução: primeiro padronizo tudo para uma forma única
(sem hífen, sem espaço extra, tudo em caixa alta) e só depois traduzo.
Assim o dicionário de tradução precisa de apenas uma entrada por status, e não de uma por variação.

In [0]:
status_normalizado = F.upper(
    F.trim(
        F.regexp_replace(
            F.regexp_replace(F.col("status"), r"[-_]+", " "),  # hífens e underlines viram espaço
            r"\s+", " "                                        # múltiplos espaços viram um só
        )
    )
)

traducao_status = {
    "RELEASED": "Lançado",
    "POST PRODUCTION": "Pós-Produção",
    "IN PRODUCTION": "Em Produção",
    "PLANNED": "Planejado",
    "RUMORED": "Rumores",
    "CANCELED": "Cancelado",
    "CANCELLED": "Cancelado",   # grafia alternativa, por segurança
}

# Qualquer valor que não esteja no dicionário (corrompido, nulo ou fora do domínio)
# cai no valor padrão "Não Informado".
coluna_status = F.coalesce(
    F.create_map([F.lit(x) for par in traducao_status.items() for x in par])[status_normalizado],
    F.lit("Não Informado")
)

### 1.3 Conversão da data de lançamento
A origem mistura três formatos. Testei cada um nos dados e confirmei qual é qual
verificando se o primeiro campo ultrapassa 12 (se ultrapassa, é dia, não mês):

| Exemplo | Formato |
|---|---|
| `2016-02-09` | `yyyy-MM-dd` |
| `04-25-2018` | `MM-dd-yyyy` |
| `16/03/2017` | `dd/MM/yyyy` |

Uso `try_to_date` dentro de um `coalesce`: ele tenta um formato por vez e só devolve NULL
quando **nenhum** dos três funciona. O `try_` é importante porque a versão comum (`to_date`)
lança exceção em valor malformado e derrubaria o pipeline inteiro por causa de 2 registros ruins.

In [0]:
coluna_data = F.coalesce(
    F.expr("try_to_date(release_date, 'yyyy-MM-dd')"),
    F.expr("try_to_date(release_date, 'MM-dd-yyyy')"),
    F.expr("try_to_date(release_date, 'dd/MM/yyyy')"),
)

### 1.4 Seleção final, renomeação e tipagem
`duracao_minutos` recebe `cast("int")`: por causa do column shift existem registros com texto
no lugar do número, e o cast os transforma em NULL sem quebrar a execução.

In [0]:
df_silver_info = df_dedup.select(
    F.col("id").cast("string").alias("id_filme"),
    F.col("title").cast("string").alias("titulo"),
    F.col("original_title").cast("string").alias("titulo_original"),
    coluna_data.alias("data_lancamento"),
    F.expr("try_cast(runtime AS INT)").alias("duracao_minutos"),
    F.col("original_language").cast("string").alias("idioma_original"),
    coluna_status.alias("status_filme"),
    F.col("overview").cast("string").alias("sinopse"),
    F.col("tagline").cast("string").alias("frase_divulgacao"),
).withColumn(
    "ano_lancamento", F.year(F.col("data_lancamento")).cast("int")  # coluna derivada
)

In [0]:
(
    df_silver_info.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_info_filmes")
)

print("silver.tb_info_filmes gravada.")

### 1.5 Validação

In [0]:
df_check = spark.table("silver.tb_info_filmes")

print(f"Total de linhas: {df_check.count()}")
print(f"Ids distintos:   {df_check.select('id_filme').distinct().count()}  (devem ser iguais)")
df_check.printSchema()

In [0]:
display(df_check.groupBy("status_filme").count().orderBy(F.desc("count")))

In [0]:
print(f"Datas que não puderam ser convertidas: {df_check.filter(F.col('data_lancamento').isNull()).count()}")
display(df_check.select("id_filme", "titulo", "data_lancamento", "ano_lancamento", "status_filme").limit(20))

## 2) silver.tb_financeiro_filmes
Origem: `bronze.tb_movies_financials`

Tratamentos aplicados:
1. Textos que representam ausência de dado viram NULL.
2. Higienização dos símbolos de moeda e pontuação, com expansão dos sufixos K/M/B.
3. Conversão para decimal; zeros e negativos tratados como ausentes.
4. Conversão para BRL usando a cotação obtida na Bronze.
5. Derivação de Lucro (USD/BRL) e Margem de Lucro Percentual.

### 2.1 Obtenção da taxa de câmbio
Os filmes são de décadas diferentes e a API do BACEN só devolve os últimos dias, então não existe data em comum para fazer um join. A conversão usa a cotação mais recente disponível, aplicada como taxa única — que é o que a diretoria financeira pediu (valores "de hoje" em BRL).

In [0]:
cotacao_dolar = (
    spark.table("bronze.tb_cotacao_dolar")
    .orderBy(F.col("dataHoraCotacao").desc())
    .select("cotacaoCompra")
    .first()[0]
)
print(f"Cotação aplicada: R$ {cotacao_dolar}")

### 2.2 Higienização das colunas monetárias
A origem mistura vários formatos na mesma coluna: `97000000`, `$ 97000000`, `USD 10000`, `34.0M`, `10.0K`, `Unknown`, `Não Informado`, nulo.

A limpeza acontece nesta ordem:
1. Texto de ausência vira NULL **antes** da conversão (senão viraria 0 ou lixo).
2. `[^0-9.KMB-]` remove tudo que não é dígito, ponto, sufixo ou sinal — some `$`, `USD`, espaço e pontuação de milhar de uma vez só.
3. O sufixo é capturado **antes** de ser removido, para virar multiplicador. Sem isso, `34.0M` viraria 34 dólares em vez de 34 milhões.
4. `try_cast` converte para DECIMAL devolvendo NULL no que for impossível, sem quebrar o pipeline.
5. Zerados e negativos viram NULL: na base de origem o zero significa "não informado", não "custou nada".

In [0]:
def higienizar_monetario(coluna_origem):
    """Monta a expressão SQL que limpa, converte e valida uma coluna monetária."""
    textos_ausentes = "'UNKNOWN','NAO INFORMADO','NÃO INFORMADO','N/A','NA','NULL','NONE','-',''"

    return f"""
        CASE
            WHEN {coluna_origem} IS NULL
              OR upper(trim({coluna_origem})) IN ({textos_ausentes})
            THEN NULL
            ELSE
                nullif(
                    greatest(
                        try_cast(
                            regexp_replace(
                                regexp_replace(upper(trim({coluna_origem})), '[^0-9.KMB-]', ''),
                                '[KMB]$', ''
                            ) AS DECIMAL(18,2)
                        )
                        *
                        CASE right(regexp_replace(upper(trim({coluna_origem})), '[^0-9.KMB-]', ''), 1)
                            WHEN 'K' THEN 1000
                            WHEN 'M' THEN 1000000
                            WHEN 'B' THEN 1000000000
                            ELSE 1
                        END,
                        0
                    ),
                    0
                )
        END
    """

### 2.3 Deduplicação
O escopo não pede explicitamente aqui, mas a camada Gold exige grão de **um registro por filme**. Se a duplicidade passar, o join com a tabela fato multiplica as linhas e infla todas as métricas.

In [0]:
df_bronze_fin = spark.table("bronze.tb_movies_financials")

janela_filme_fin = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())

df_fin_dedup = (
    df_bronze_fin
    .withColumn("num_versao", F.row_number().over(janela_filme_fin))
    .filter(F.col("num_versao") == 1)
    .drop("num_versao")
)

print(f"Linhas na Bronze:        {df_bronze_fin.count()}")
print(f"Linhas após deduplicar:  {df_fin_dedup.count()}")

### 2.4 Conversão para BRL e colunas derivadas
O `coalesce(coluna, 0)` no cálculo do lucro atende à regra de que valores ausentes não podem invalidar o resultado: sem ele, um orçamento nulo faria o lucro inteiro virar NULL.

A margem só é calculada quando a receita é maior que zero — é assim que se evita a divisão por zero sem precisar tratar exceção.

In [0]:
df_silver_fin = (
    df_fin_dedup
    .select(
        F.col("id").cast("string").alias("id_filme"),
        F.expr(higienizar_monetario("budget")).alias("orcamento_usd"),
        F.expr(higienizar_monetario("revenue")).alias("receita_usd"),
    )
    .withColumn("orcamento_brl", (F.col("orcamento_usd") * F.lit(cotacao_dolar)).cast("decimal(18,2)"))
    .withColumn("receita_brl",   (F.col("receita_usd")   * F.lit(cotacao_dolar)).cast("decimal(18,2)"))
    .withColumn("lucro_usd", (F.coalesce(F.col("receita_usd"), F.lit(0)) - F.coalesce(F.col("orcamento_usd"), F.lit(0))).cast("decimal(18,2)"))
    .withColumn("lucro_brl", (F.coalesce(F.col("receita_brl"), F.lit(0)) - F.coalesce(F.col("orcamento_brl"), F.lit(0))).cast("decimal(18,2)"))
    .withColumn(
        "margem_lucro_percentual",
        F.when(
            F.col("receita_usd") > 0,
            F.round((F.col("lucro_usd") / F.col("receita_usd")) * 100, 2)
        ).otherwise(F.lit(None)).cast("decimal(18,2)")
    )
)

In [0]:
(
    df_silver_fin.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_financeiro_filmes")
)

print("silver.tb_financeiro_filmes gravada.")

### 2.5 Validação

In [0]:
df_check_fin = spark.table("silver.tb_financeiro_filmes")

print(f"Total de linhas: {df_check_fin.count()}")
print(f"Ids distintos:   {df_check_fin.select('id_filme').distinct().count()}  (devem ser iguais)")
df_check_fin.printSchema()

In [0]:
display(
    df_check_fin.agg(
        F.count("*").alias("linhas"),
        F.count("orcamento_usd").alias("orcamento_preenchido"),
        F.count("receita_usd").alias("receita_preenchida"),
        F.min("orcamento_usd").alias("menor_orcamento"),
        F.min("receita_usd").alias("menor_receita"),
        F.max("receita_usd").alias("maior_receita"),
    )
)

In [0]:
display(
    df_check_fin
    .filter(F.col("receita_usd").isNotNull() & F.col("orcamento_usd").isNotNull())
    .orderBy(F.desc("receita_usd"))
    .limit(15)
)

## 3) silver.tb_metricas_engajamento
Origem: `bronze.tb_movies_metrics`

Tratamentos aplicados:
1. Deduplicação por filme, mantendo a ingestão mais recente.
2. Limpeza do separador decimal da popularidade (vírgula → ponto).
3. Conversão de tipagem segura: textos deslocados por Column Shift viram NULL sem quebrar o pipeline.
4. Regras de negócio: notas fora do intervalo 0–10 e valores negativos são invalidados.

### 3.1 Limpeza da popularidade
A origem usa vírgula como separador **decimal** (`0,6` = zero vírgula seis), e não como separador de milhar — é o oposto do caso das colunas monetárias.

Por isso a vírgula é **substituída por ponto**, e não apagada: apagar transformaria `0,6` em `06`, um valor dez vezes maior.

In [0]:
df_bronze_met = spark.table("bronze.tb_movies_metrics")

janela_filme_met = Window.partitionBy("id").orderBy(F.col("ingestion_datetime").desc())

df_met_dedup = (
    df_bronze_met
    .withColumn("num_versao", F.row_number().over(janela_filme_met))
    .filter(F.col("num_versao") == 1)
    .drop("num_versao")
)

print(f"Linhas na Bronze:        {df_bronze_met.count()}")
print(f"Linhas após deduplicar:  {df_met_dedup.count()}")

### 3.2 Conversão segura e regras de negócio
O Column Shift espalhou nomes de diretores, idiomas e países pelas colunas numéricas (`Tomohito Oosaki` em `vote_average`, `English` em `numVotes`). O `try_cast` devolve NULL nesses casos em vez de interromper a execução.

Depois da conversão entram os limites de negócio:
- Notas fora de 0–10 viram NULL. Existem 2.998 registros com nota em escala 0–100 (`60.0`, `100.0`); o escopo manda desconsiderar, não corrigir dividindo por 10.
- Contagens de votos e popularidade negativas viram NULL.

In [0]:
# Popularidade: troca o separador decimal e remove qualquer caractere que não seja dígito, ponto ou sinal.
popularidade_limpa = F.expr("""
    try_cast(regexp_replace(trim(popularity), ',', '.') AS DOUBLE)
""")

# Conversão de tipagem segura para decimais e inteiros
def nota_valida(coluna):
    """Converte para DOUBLE e anula o que estiver fora do intervalo de negócio 0 a 10."""
    return F.expr(f"""
        CASE
            WHEN try_cast({coluna} AS DOUBLE) BETWEEN 0 AND 10
            THEN try_cast({coluna} AS DOUBLE)
            ELSE NULL
        END
    """)

def contagem_valida(coluna):
    """Converte para INT e anula valores negativos."""
    return F.expr(f"""
        CASE
            WHEN try_cast({coluna} AS INT) >= 0
            THEN try_cast({coluna} AS INT)
            ELSE NULL
        END
    """)

In [0]:
df_silver_met = df_met_dedup.select(
    F.col("id").cast("string").alias("id_filme"),
    F.when(popularidade_limpa >= 0, popularidade_limpa).alias("popularidade"),
    nota_valida("vote_average").alias("nota_media_tmdb"),
    contagem_valida("vote_count").alias("qtd_votos_tmdb"),
    nota_valida("averageRating").alias("nota_media_imdb"),
    contagem_valida("numVotes").alias("qtd_votos_imdb"),
)

In [0]:
(
    df_silver_met.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.tb_metricas_engajamento")
)

print("silver.tb_metricas_engajamento gravada.")

### 3.3 Validação

In [0]:
df_check_met = spark.table("silver.tb_metricas_engajamento")

print(f"Total de linhas: {df_check_met.count()}")
print(f"Ids distintos:   {df_check_met.select('id_filme').distinct().count()}  (devem ser iguais)")
df_check_met.printSchema()

In [0]:
# Notas devem ser de 0 a 10
display(
    df_check_met.agg(
        F.count("popularidade").alias("popularidade_ok"),
        F.count("nota_media_tmdb").alias("nota_tmdb_ok"),
        F.count("nota_media_imdb").alias("nota_imdb_ok"),
        F.min("popularidade").alias("min_popularidade"),
        F.max("nota_media_tmdb").alias("max_nota_tmdb"),
        F.max("nota_media_imdb").alias("max_nota_imdb"),
        F.min("qtd_votos_tmdb").alias("min_votos_tmdb"),
        F.min("qtd_votos_imdb").alias("min_votos_imdb"),
    )
)

In [0]:
display(df_check_met.orderBy(F.desc("popularidade")).limit(15))